基于 gemini 引导重新输入对应的 Ta 来计算相关的 reward 查看数值结果，核查是否可以引导

## 单目标设计

In [ ]:
import numpy as np

def calculate_T_a_reward(T_a, T_a_reference=1.5, T_a_lower_bound=1.128):
    """
    计算 T_a 变量的奖励函数。

    Args:
        T_a (float): 当前环境的 T_a 变量值。
        T_a_reference (float): T_a 的参考值（橙线）。
        T_a_lower_bound (float): T_a 的下限，低于此值将受到惩罚。

    Returns:
        float: 计算得到的奖励值。
    """

    reward = 0.0

    # 可以根据您的需求调整这些参数
    reward_scale_factor_below = 10.0  # T_a低于参考值时，距离越远奖励越大
    penalty_scale_factor_above = 5.0 # T_a高于参考值时，距离越远惩罚越大
    penalty_for_too_low = -20.0       # T_a低于下限时的固定惩罚

    # 1. T_a 低于 T_a_lower_bound 时的惩罚
    if T_a < T_a_lower_bound:
        reward = penalty_for_too_low
        # 也可以考虑惩罚与距离下限的差值挂钩，例如：
        # reward = penalty_for_too_low - (T_a_lower_bound - T_a) * some_other_penalty_factor
        # 这里为了简洁和明确，先给一个固定大惩罚。
        print(f"DEBUG: T_a ({T_a:.3f}) is too low, applying fixed penalty: {reward:.3f}")
        return reward # 如果太低了，直接返回惩罚，不考虑其他情况

    # 2. T_a 在 T_a_lower_bound 和 T_a_reference 之间 (理想情况)
    elif T_a_lower_bound <= T_a < T_a_reference:
        # 目标是 T_a 尽量低于 T_a_reference，且越远越好
        # 因此，距离 T_a_reference 越远 (即 T_a 越小)，奖励越高。
        reward = (T_a_reference - T_a) * reward_scale_factor_below
        print(f"DEBUG: T_a ({T_a:.3f}) is in desired range, reward: {reward:.3f}")
        
    # 3. T_a 高于或等于 T_a_reference (允许越界，但惩罚)
    else: # T_a >= T_a_reference
        # 惩罚与超出参考值的距离成正比
        penalty = (T_a - T_a_reference) * penalty_scale_factor_above
        reward = -penalty # 奖励为负值
        print(f"DEBUG: T_a ({T_a:.3f}) exceeded reference, penalty: {penalty:.3f}, reward: {reward:.3f}")

    return reward

# --- 测试示例 ---
print("--- 测试 T_a 奖励函数 ---")

# 理想情况：低于参考值，且高于下限
print(f"T_a = 1.3: Reward = {calculate_T_a_reward(1.3):.3f}")  # 期望正值，且比1.45高
print(f"T_a = 1.45: Reward = {calculate_T_a_reward(1.45):.3f}") # 期望正值，但比1.3低

# 刚好在参考值
print(f"T_a = 1.5: Reward = {calculate_T_a_reward(1.5):.3f}")   # 期望负值（微小惩罚）

# 越界但不太高
print(f"T_a = 1.55: Reward = {calculate_T_a_reward(1.55):.3f}")  # 期望负值
print(f"T_a = 1.6: Reward = {calculate_T_a_reward(1.6):.3f}")    # 期望更大的负值

# 远高于参考值
print(f"T_a = 1.8: Reward = {calculate_T_a_reward(1.8):.3f}")    # 期望很大的负值

# 低于下限
print(f"T_a = 1.1: Reward = {calculate_T_a_reward(1.1):.3f}")    # 期望大的负值
print(f"T_a = 1.127: Reward = {calculate_T_a_reward(1.127):.3f}") # 期望大的负值
print(f"T_a = 1.128: Reward = {calculate_T_a_reward(1.128):.3f}") # 期望正值（刚好处在理想区间）

--- 测试 T_a 奖励函数 ---
DEBUG: T_a (1.300) is in desired range, reward: 2.000
T_a = 1.3: Reward = 2.000
DEBUG: T_a (1.450) is in desired range, reward: 0.500
T_a = 1.45: Reward = 0.500
DEBUG: T_a (1.500) exceeded reference, penalty: 0.000, reward: -0.000
T_a = 1.5: Reward = -0.000
DEBUG: T_a (1.550) exceeded reference, penalty: 0.250, reward: -0.250
T_a = 1.55: Reward = -0.250
DEBUG: T_a (1.600) exceeded reference, penalty: 0.500, reward: -0.500
T_a = 1.6: Reward = -0.500
DEBUG: T_a (1.800) exceeded reference, penalty: 1.500, reward: -1.500
T_a = 1.8: Reward = -1.500
DEBUG: T_a (1.100) is too low, applying fixed penalty: -20.000
T_a = 1.1: Reward = -20.000
DEBUG: T_a (1.127) is too low, applying fixed penalty: -20.000
T_a = 1.127: Reward = -20.000
DEBUG: T_a (1.128) is in desired range, reward: 3.720
T_a = 1.128: Reward = 3.720
